# Stage 1.2 Horizon Calibration

Empirical calibration of the maximum triple-barrier holding horizon using 2010-2011 entries.

## Setup

Import the production calibration script as a library and configure paths.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8")
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.calibrate_horizon import calibrate_horizon, compute_atr_wilder, write_horizon_yaml

MASTER_PATH = REPO_ROOT / "data" / "processed" / "master_dataset.parquet"
OUTPUT_PATH = REPO_ROOT / "config" / "horizon.yaml"

## 1. Load Master Dataset

Load the full master dataset. ATR uses pre-2010 rows for warm-up; simulations only open entries in 2010-2011 and use 2012 as lookout buffer where needed.

In [ ]:
master_df = pd.read_parquet(MASTER_PATH).sort_index()
warmup_and_calibration = master_df.loc["2009-12-01":"2011-12-31"]
lookout_buffer = master_df.loc["2012-01-01":"2012-03-31"]

display(pd.DataFrame({
    "rows": [len(master_df)],
    "start": [master_df.index.min()],
    "end": [master_df.index.max()],
    "warmup_calibration_rows": [len(warmup_and_calibration)],
    "lookout_buffer_rows": [len(lookout_buffer)],
}))

## 2. Compute ATR(14) Wilder

ATR is computed on the full history, then shifted by one day so entry-day barriers only use information available through `t-1`.

In [ ]:
atr = compute_atr_wilder(
    master_df["us100_high"],
    master_df["us100_low"],
    master_df["us100_close"],
    period=14,
)
entry_atr = atr.shift(1)
calibration_atr = entry_atr.loc["2010-01-01":"2011-12-31"]

assert not calibration_atr.isna().any()
display(calibration_atr.describe().to_frame("entry_atr"))

## 3. Run Calibration

Simulate the static TP/SL barriers for every 2010-2011 entry day.

In [ ]:
result = calibrate_horizon(master_df)
simulations = pd.DataFrame(result["simulation_results"])
hit_times = simulations.loc[~simulations["no_barrier_hit"], "time_to_first_barrier"].astype(int)

summary = pd.DataFrame({
    "metric": ["horizon_days", "n_simulations", "n_barrier_hit", "n_no_barrier_hit", "pct_no_barrier_hit"],
    "value": [result["horizon_days"], result["n_simulations"], result["n_barrier_hit"],
              result["n_no_barrier_hit"], result["pct_no_barrier_hit"]],
})
display(summary)

## 4. Plot Time-To-First-Barrier Distribution

Histogram of hit times, with mean, median, and calibrated 75th percentile.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(hit_times, bins=range(1, 62), edgecolor="white", alpha=0.85)
ax.axvline(result["distribution_stats"]["p75"], color="crimson", linestyle="--", label="p75")
ax.axvline(result["distribution_stats"]["mean"], color="black", linestyle=":", label="mean")
ax.axvline(result["distribution_stats"]["median"], color="darkgreen", linestyle="-.", label="median")
ax.set_title("Time to first barrier, hit cases only")
ax.set_xlabel("Trading days")
ax.set_ylabel("Count")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Plot TP vs SL Split

Show which barrier was hit first among positions that resolved within the lookout window.

In [ ]:
split = result["tp_vs_sl_split"]
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    [split["pct_hit_tp_first"], split["pct_hit_sl_first"]],
    labels=["TP first", "SL first"],
    autopct="%.1f%%",
    startangle=90,
)
ax.set_title("First barrier split")
plt.show()

## 6. Diagnostics Table

Exact distribution diagnostics used for `config/horizon.yaml`.

In [ ]:
diagnostics = pd.DataFrame([
    {"group": "distribution", **result["distribution_stats"]},
    {"group": "tp_vs_sl", **result["tp_vs_sl_split"]},
])
display(diagnostics.T)
display(simulations.head())

## 7. Summary

Horizon = **14 trading days**. This is the 75th percentile of `time_to_first_barrier` among 503 barrier-hit cases from 504 simulations. A 250-day test slice can fit roughly `250 / 14 = 17.9` non-overlapping horizon windows, though later stages may allow overlapping labels with embargo logic in Stage 1.4.

## 8. Save Horizon YAML

Persist the calibrated value for Stage 1.3 labels and Stage 1.7 backtest assumptions.

In [ ]:
write_horizon_yaml(result, OUTPUT_PATH)
print(f"Wrote {OUTPUT_PATH}")